# PennyLane finite shots

Compare finite-shot Bell counts from default.qubit and MettleQ while keeping exact equality separate from statistical agreement.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

Finite-shot QNodes return sampled counts. Even equal distributions usually yield different count dictionaries.

In [2]:
shots = 4096
def make_counts(device):
    @qml.qnode(device)
    def circuit():
        qml.Hadamard(0)
        qml.CNOT(wires=[0, 1])
        return qml.counts(wires=[0, 1])
    return circuit

reference_device = qml.device("default.qubit", wires=2, shots=shots, seed=27)
reference_qnode = make_counts(reference_device)

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_qnode)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=2, shots=shots, seed=27, method="statevector", device="cpu")
mettleq_qnode = make_counts(mettleq_device)
candidate, mettleq_ms, _ = benchmark(mettleq_qnode)
tvd = total_variation_distance(reference, candidate)
support_ok = set(reference) <= {"00", "11"} and set(candidate) <= {"00", "11"}
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

The observed supports and total-variation distance are checked rather than requiring identical random samples.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/02_finite_shots.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="finite-shot total-variation distance <= 0.05",
    passed=support_ok and tvd <= 0.05,
    exact_match=reference == candidate,
    selected_method=method,
    selected_device=device,
    metrics={"tvd": tvd, "reference_counts": reference, "mettleq_counts": candidate},
    notes="Independent device RNG implementations need not return identical count dictionaries.",
)


Comparison summary
------------------
Correctness contract: PASS — finite-shot total-variation distance <= 0.05
SDK reference median: 6.041 ms
MettleQ median:       6.429 ms
Timing interpretation: the SDK reference was 1.064x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)
Note: Independent device RNG implementations need not return identical count dictionaries.

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "finite-shot total-variation distance <= 0.05", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"mettleq_counts": {"00": 2027, "11": 2069}, "reference_counts": {"00": 2020, "11": 2076}, "tvd": 0.001708984375}, "mettleq_median_ms": 6.429249973734841, "notebook": "pennylane/02_finite_shots.ipynb", "notes": "Independent device RNG implementations need not return identical count dictionaries.", "passed": true, "python": "3.13.2", "re

## What should you conclude?

Choose this contract when training or analysis consumes shot noise; speed depends strongly on shot count and batching.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.